<a href="https://colab.research.google.com/github/Shemuel-bot/AI-ball-notebooks/blob/main/ai_ball_steps_kaggle_ga_loss_decimal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Ball — Genetic Algorithm with Relaxed-Brain Loss

This notebook keeps only the code needed to evolve a `Ball_Trap_Brain` with a genetic algorithm.

The GA's fitness is the **same loss objective used by the relaxed brain**:
- first-hole output distributions are scored with cross-entropy;
- every trajectory position is included, with an explicit **emit-nothing** class;
- target characters are right-aligned to trajectory positions;
- the repeat/self-loop penalty is included;
- lower loss is better, so the GA selects the individual with the **lowest loss**.

`temperature` is a direct GA parameter. It controls the softness of the discrete-to-relaxed representation used when calculating fitness. Lower values make the fitness calculation closer to the discrete brain; higher values make it smoother.


In [ ]:
import math
import random
import torch


def TokenizeInput(input_value, alphabet):
    """Tokenize a string using the same longest-first, non-overlapping rule."""
    input_str = str(input_value)
    matches = []

    for order, item in enumerate(alphabet):
        item_str = str(item)
        if not item_str:
            continue

        start_pos = 0
        while True:
            index = input_str.find(item_str, start_pos)
            if index == -1:
                break
            matches.append((index, -len(item_str), order, item))
            start_pos = index + 1

    matches.sort(key=lambda match: (match[0], match[1], match[2]))

    result = []
    last_end = 0
    for start_index, neg_length, _, item in matches:
        length = -neg_length
        if start_index >= last_end:
            result.append(item)
            last_end = start_index + length

    return result


class Ball_Trap_Brain:
    def __init__(
        self,
        deflectors,
        bounce,
        alphabet,
        output,
        holes_multiplier=1,
        direction_range=3,
    ):
        if deflectors < 1:
            raise ValueError("deflectors must be >= 1")
        if not alphabet:
            raise ValueError("alphabet must not be empty")
        if not output:
            raise ValueError("output must not be empty")
        if direction_range < 0:
            raise ValueError("direction_range must be >= 0")

        self.output = list(output)
        self.holes_multiplier = holes_multiplier
        self.ball_energy = bounce * 2
        self.alphabet = list(alphabet)
        self.direction_range = direction_range

        self.alphabet_directions = self.choose_alphabet_direction(len(self.alphabet))
        self.alphabet_start_positions = self.start_positions(
            len(self.alphabet), deflectors
        )
        self.deflectors_list = [
            [random.randint(0, deflectors - 1), bounce]
            for _ in range(deflectors)
        ]
        self.set_holes(self.output, deflectors, self.holes_multiplier)

    def set_holes(self, output, deflectors, holes_multiplier):
        for symbol in output:
            for _ in range(holes_multiplier):
                self.deflectors_list[random.randrange(deflectors)][1] = symbol

    def choose_alphabet_direction(self, alphabet_size):
        return [
            random.randint(-self.direction_range, self.direction_range)
            for _ in range(alphabet_size)
        ]

    def start_positions(self, alphabet_size, deflectors):
        return [random.randrange(deflectors) for _ in range(alphabet_size)]

    def compute_input(self, input_value):
        alphabet_index = {token: i for i, token in enumerate(self.alphabet)}
        tokens = TokenizeInput(input_value, self.alphabet)

        directions = []
        positions = []
        for token in tokens:
            idx = alphabet_index.get(token)
            if idx is not None:
                directions.append(self.alphabet_directions[idx])
                positions.append(self.alphabet_start_positions[idx])

        travel = [
            self.travel(directions.copy(), positions[i], i)
            for i in range(len(directions))
        ]
        return self.ricochet(travel, cap=10, ball_energy=self.ball_energy)

    def ricochet(self, travel, cap, ball_energy):
        result = []
        dl = self.deflectors_list
        n = len(dl)

        for t in travel:
            new_idx = t % n
            energy = ball_energy

            for _ in range(cap):
                if energy <= 0:
                    break

                dest, value = dl[new_idx]
                distance = abs(dest - new_idx)

                if not isinstance(value, int):
                    result.append(value)
                    energy += 1
                    energy -= distance
                else:
                    energy += value
                    energy -= distance

                new_idx = dest

        return result

    def travel(self, directions, position, index):
        # Exactly matches the original weighted-position travel rule.
        total = 0
        for j, direction in enumerate(directions):
            if j != index:
                total += direction * (j + 1)
        return position + total

    def clone(self):
        new_brain = object.__new__(Ball_Trap_Brain)
        new_brain.output = self.output[:]
        new_brain.holes_multiplier = self.holes_multiplier
        new_brain.ball_energy = self.ball_energy
        new_brain.alphabet = self.alphabet[:]
        new_brain.direction_range = self.direction_range
        new_brain.deflectors_list = [[d, v] for d, v in self.deflectors_list]
        new_brain.alphabet_directions = self.alphabet_directions[:]
        new_brain.alphabet_start_positions = self.alphabet_start_positions[:]
        return new_brain

    def Mutate(self, individual_probability=1.0, probability=0.025):
        """Apply the same mutation style used by the original GA."""
        if random.random() >= individual_probability:
            return self

        n = len(self.deflectors_list)

        for i in range(n):
            if random.random() < probability:
                self.deflectors_list[i][0] = random.randrange(n)

        for _ in range(n):
            if random.random() < probability:
                idx = random.randrange(n)
                self.deflectors_list[idx][1] = random.choice(self.output)

        for i in range(len(self.alphabet_directions)):
            if random.random() < probability:
                self.alphabet_directions[i] = random.randint(
                    -self.direction_range, self.direction_range
                )

        for i in range(len(self.alphabet_start_positions)):
            if random.random() < probability:
                self.alphabet_start_positions[i] = random.randrange(n)

        if random.random() < probability:
            self.set_holes(self.output, n, self.holes_multiplier)

        return self

    # ------------------------------------------------------------------
    # Relaxed-brain forward pass used ONLY for GA fitness.
    #
    # The loss below mirrors the relaxed trainer's objective. To make it
    # usable for discrete GA candidates without constructing an O(D^2)
    # transition matrix, the categorical softmaxes are evaluated through
    # their algebraically equivalent "one selected logit + equal baseline"
    # form.
    # ------------------------------------------------------------------

    @staticmethod
    def _selected_softmax_constants(size, temperature, logit_strength=1.0):
        if size < 1:
            raise ValueError("categorical size must be >= 1")
        if temperature <= 0:
            raise ValueError("temperature must be > 0")

        if size == 1:
            return 1.0, 0.0

        x = logit_strength / temperature
        e = math.exp(-x)  # stable even when x is very large
        p = 1.0 / (1.0 + (size - 1) * e)
        q = e * p
        return p, q

    def _relaxed_parameters(self, temperature, logit_strength=1.0):
        """
        Build the quantities needed by the relaxed loss from this discrete brain.

        A discrete choice is embedded as:
            selected logit = logit_strength
            all other logits = 0
        followed by softmax(... / temperature).

        This gives the GA a deterministic, temperature-controlled relaxed
        representation of each candidate brain.
        """
        D = len(self.deflectors_list)
        A = len(self.alphabet)
        K = len(self.output)
        V = self.direction_range

        if temperature <= 0:
            raise ValueError("temperature must be > 0")

        output_index = {symbol: i for i, symbol in enumerate(self.output)}

        dest = torch.tensor(
            [int(row[0]) for row in self.deflectors_list],
            dtype=torch.long,
        )
        is_hole = torch.tensor(
            [not isinstance(row[1], int) for row in self.deflectors_list],
            dtype=torch.bool,
        )
        hole_p = torch.where(
            is_hole,
            torch.full((D,), torch.sigmoid(torch.tensor(1.0)).item()),
            torch.full((D,), torch.sigmoid(torch.tensor(-1.0)).item()),
        )

        symbol_index = torch.tensor(
            [
                output_index[row[1]]
                if row[1] in output_index
                else 0
                for row in self.deflectors_list
            ],
            dtype=torch.long,
        )
        # Bounce values are copied exactly from the discrete brain.
        bounce_val = torch.tensor(
            [
                float(row[1]) if isinstance(row[1], int) else 0.0
                for row in self.deflectors_list
            ],
            dtype=torch.float32,
        )

        direction_values = torch.arange(-V, V + 1, dtype=torch.float32)
        direction_choices = torch.tensor(
            [
                max(-V, min(V, int(direction)))
                for direction in self.alphabet_directions
            ],
            dtype=torch.long,
        ) + V

        start_choices = torch.tensor(
            self.alphabet_start_positions,
            dtype=torch.long,
        )

        dest_p, dest_q = self._selected_softmax_constants(
            D, temperature, logit_strength
        )
        symbol_p, symbol_q = self._selected_softmax_constants(
            K, temperature, logit_strength
        )
        direction_p, direction_q = self._selected_softmax_constants(
            len(direction_values), temperature, logit_strength
        )
        start_p, start_q = self._selected_softmax_constants(
            D, temperature, logit_strength
        )

        # Expected direction and expected start under the same relaxed softmax
        # construction used by the other categorical parameters.
        dir_sum = direction_values.sum().item()
        expected_direction = (
            direction_q * dir_sum
            + (direction_p - direction_q) * direction_values[direction_choices]
        )

        pos = torch.arange(D, dtype=torch.float32)
        pos_sum = pos.sum().item()
        expected_start = (
            start_q * pos_sum
            + (start_p - start_q) * pos[start_choices]
        )

        # Expected hop distance for each currently occupied deflector.
        # q * sum_j |j-d| + (p-q) * |dest[d]-d|.
        # sum_j |j-d| has a closed form, avoiding an O(D^2) temporary matrix.
        d_float = torch.arange(D, dtype=torch.float32)
        row_abs_sum = (
            d_float * (d_float + 1.0) / 2.0
            + (D - 1.0 - d_float) * (D - d_float) / 2.0
        )
        weighted_abs = (
            dest_q * row_abs_sum
            + (dest_p - dest_q) * torch.abs(dest.float() - pos)
        )

        return {
            "D": D,
            "A": A,
            "K": K,
            "dest": dest,
            "hole_p": hole_p,
            "is_hole": is_hole,
            "symbol_index": symbol_index,
            "bounce_val": bounce_val,
            "expected_direction": expected_direction,
            "expected_start": expected_start,
            "weighted_abs": weighted_abs,
            "dest_p": dest_p,
            "dest_q": dest_q,
            "symbol_p": symbol_p,
            "symbol_q": symbol_q,
        }

    def _relaxed_soft_index_dist(self, travel_values, temperature, sharpness):
        D = len(self.deflectors_list)
        positions = torch.arange(D, dtype=torch.float32)
        diff = positions.unsqueeze(0) - travel_values.unsqueeze(1)
        diff = torch.remainder(diff + D / 2.0, D) - D / 2.0

        logits = -(diff ** 2) / (sharpness * temperature)
        return torch.softmax(logits, dim=1)

    def _relaxed_simulate(
        self,
        input_indices,
        temperature,
        sharpness=0.75,
        logit_strength=1.0,
        cap=10,
        params=None,
    ):
        """
        Differentiable-relaxation-style simulation, evaluated without gradients.

        Returns one output distribution, never-fire probability, and
        repeat-fire expectation for each input token.
        """
        p = (
            params
            if params is not None
            else self._relaxed_parameters(temperature, logit_strength)
        )

        if not input_indices:
            return [], [], []

        dirs = p["expected_direction"][input_indices]
        starts = p["expected_start"][input_indices]

        L = len(input_indices)
        position_weights = torch.arange(1, L + 1, dtype=torch.float32)
        weighted_dirs = dirs * position_weights
        total_weighted = weighted_dirs.sum()
        travel = starts + (total_weighted - weighted_dirs)

        state = self._relaxed_soft_index_dist(
            travel, temperature, sharpness
        )  # (L, D)

        D = p["D"]
        K = p["K"]
        dest = p["dest"]
        hole_p = p["hole_p"]
        symbol_index = p["symbol_index"]
        bounce_val = p["bounce_val"]

        energy = torch.full(
            (L,), float(self.ball_energy), dtype=torch.float32
        )
        alive = torch.ones(L, dtype=torch.float32)
        not_hole_yet = torch.ones(L, dtype=torch.float32)
        out_dist = torch.zeros(L, K, dtype=torch.float32)
        extra = torch.zeros(L, dtype=torch.float32)

        for _ in range(cap):
            p_hole = (state * hole_p).sum(dim=1)
            p_bounce_energy = (
                state * (1.0 - hole_p) * bounce_val
            ).sum(dim=1)
            exp_dist = (state * p["weighted_abs"]).sum(dim=1)

            already_fired = alive * (1.0 - not_hole_yet)
            extra += already_fired * p_hole

            p_first_hole = alive * not_hole_yet * p_hole

            # Relaxed expected output distribution:
            # q for every symbol + (p-q) on each deflector's selected symbol.
            # Only actual hole rows have a selected output symbol in the
            # relaxed parameterization. Bounce rows keep a uniform symbol
            # distribution, exactly as in the original relaxed brain.
            weighted_holes = state * hole_p
            hole_selected_mass = weighted_holes * p["is_hole"].unsqueeze(0)

            symbol_bucket = torch.zeros(
                (L, K), dtype=torch.float32
            )
            symbol_bucket.scatter_add_(
                1,
                symbol_index.unsqueeze(0).expand(L, -1),
                hole_selected_mass,
            )

            bounce_hole_mass = (
                p_hole - hole_selected_mass.sum(dim=1)
            )
            symbol_numerator = (
                (bounce_hole_mass / K).unsqueeze(1)
                + p["symbol_q"] * hole_selected_mass.sum(dim=1).unsqueeze(1)
                + (p["symbol_p"] - p["symbol_q"]) * symbol_bucket
            )
            sym_dist = symbol_numerator
            # Match the relaxed brain exactly: symbol probabilities are
            # conditional on being at a hole.
            sym_dist = sym_dist / (p_hole.unsqueeze(1) + 1e-8)

            out_dist += p_first_hole.unsqueeze(1) * sym_dist

            energy = energy + p_hole + p_bounce_energy - exp_dist
            alive = alive * torch.sigmoid(energy * 2.0)
            not_hole_yet = not_hole_yet * (1.0 - p_hole)

            # Equivalent to state @ dest_probs without materializing the
            # D x D matrix. Each trajectory remains independent.
            routed = torch.zeros_like(state)
            routed.scatter_add_(
                1,
                dest.unsqueeze(0).expand(L, -1),
                state,
            )
            state = (
                p["dest_q"] * state.sum(dim=1, keepdim=True)
                + (p["dest_p"] - p["dest_q"]) * routed
            )

        never_fired = 1.0 - out_dist.sum(dim=1)

        return (
            list(out_dist.unbind(dim=0)),
            list(never_fired.unbind(dim=0)),
            list(extra.unbind(dim=0)),
        )

    def relaxed_loss(
        self,
        test_cases,
        temperature=0.05,
        sharpness=0.75,
        logit_strength=1.0,
        repeat_penalty_weight=0.3,
        cap=10,
        loss_decimal_places=None,
    ):
        """
        Return the relaxed brain's task loss, plus its repeat-fire penalty.

        If ``loss_decimal_places`` is set, the returned loss is rounded to
        that many decimal places. This controls the precision actually used
        by the GA fitness function.
        """
        if not test_cases:
            raise ValueError("test_cases must not be empty")
        if loss_decimal_places is not None:
            if not isinstance(loss_decimal_places, int) or isinstance(loss_decimal_places, bool):
                raise TypeError("loss_decimal_places must be an integer or None")
            if loss_decimal_places < 0:
                raise ValueError("loss_decimal_places must be >= 0 or None")

        output_index = {symbol: i for i, symbol in enumerate(self.output)}
        alphabet_index = {token: i for i, token in enumerate(self.alphabet)}
        K = len(self.output)

        relaxed_params = self._relaxed_parameters(
            temperature=temperature,
            logit_strength=logit_strength,
        )

        total_loss = 0.0
        count = 0
        repeat_loss = 0.0
        repeat_count = 0

        # The relaxed parameters only depend on the candidate brain and
        # temperature, so build them once and reuse them over the cases.
        # _relaxed_simulate currently rebuilds the small parameter object;
        # keeping this method simple and side-effect-free is preferable for
        # GA correctness. Population-level parallelism can be added later
        # without changing the objective.
        for input_value, target in test_cases:
            tokens = TokenizeInput(input_value, self.alphabet)
            indices = [
                alphabet_index[token]
                for token in tokens
                if token in alphabet_index
            ]
            if not indices:
                continue

            outs, never_fired, extra_fire = self._relaxed_simulate(
                indices,
                temperature=temperature,
                sharpness=sharpness,
                logit_strength=logit_strength,
                cap=cap,
                params=relaxed_params,
            )

            target_indices = [
                output_index[ch]
                for ch in str(target)
                if ch in output_index
            ]

            L = len(outs)
            n = min(L, len(target_indices))
            out_offset = L - n
            tgt_offset = len(target_indices) - n

            target_class = (
                [K] * out_offset
                + [
                    target_indices[tgt_offset + (i - out_offset)]
                    for i in range(out_offset, L)
                ]
            )

            outs_stack = torch.stack(outs, dim=0)
            never_stack = torch.stack(never_fired, dim=0)
            target_tensor = torch.tensor(target_class, dtype=torch.long)

            is_blank = target_tensor == K
            safe_idx = target_tensor.clamp(max=K - 1)
            gathered = outs_stack.gather(
                1, safe_idx.unsqueeze(1)
            ).squeeze(1)
            target_prob = torch.where(
                is_blank,
                never_stack,
                gathered,
            )

            total_loss += (
                -torch.log(target_prob + 1e-8).sum().item()
            )
            count += L

            repeat_loss += torch.stack(extra_fire).sum().item()
            repeat_count += L

        if count == 0:
            raise ValueError(
                "No usable test cases contained tokens from the brain alphabet."
            )

        task_loss = total_loss / count
        combined_loss = task_loss
        if repeat_penalty_weight and repeat_count:
            combined_loss += (
                repeat_penalty_weight * repeat_loss / repeat_count
            )

        combined_loss = float(combined_loss)
        if loss_decimal_places is not None:
            combined_loss = round(combined_loss, loss_decimal_places)

        return combined_loss


def relaxed_loss_fitness(
    brain,
    test_cases,
    temperature=0.05,
    sharpness=0.75,
    logit_strength=1.0,
    repeat_penalty_weight=0.3,
    loss_decimal_places=None,
):
    """Standalone fitness wrapper: the GA minimizes this returned loss."""
    return brain.relaxed_loss(
        test_cases=test_cases,
        temperature=temperature,
        sharpness=sharpness,
        logit_strength=logit_strength,
        repeat_penalty_weight=repeat_penalty_weight,
        loss_decimal_places=loss_decimal_places,
    )


In [ ]:
def genetic_algorithm(
    test_cases,
    *,
    alphabet,
    output,
    deflectors=1000,
    bounce=4,
    holes_multiplier=2,
    direction_range=3,
    generations=500,
    population_size=100,
    individual_probability=1.0,
    mutation_probability=0.025,
    temperature=0.05,
    sharpness=0.75,
    logit_strength=1.0,
    repeat_penalty_weight=0.3,
    loss_decimal_places=6,
    elite_count=1,
    seed=None,
    initial_brain=None,
    target_loss=0.0,
    verbose=True,
):
    """
    Evolve a Ball_Trap_Brain using the relaxed-brain loss as GA fitness.

    Fitness convention:
        loss = relaxed_loss_fitness(...)
        lower loss = better individual

    Temperature is intentionally exposed as a direct GA parameter. It is not
    changed automatically during generations.

    loss_decimal_places controls how many decimal places are retained in the
    relaxed loss before candidates are compared. Lower loss is still better.
    """
    if not test_cases:
        raise ValueError("test_cases must not be empty")
    if population_size < 1:
        raise ValueError("population_size must be >= 1")
    if generations < 1:
        raise ValueError("generations must be >= 1")
    if elite_count < 1 or elite_count > population_size:
        raise ValueError("elite_count must be between 1 and population_size")
    if temperature <= 0:
        raise ValueError("temperature must be > 0")
    if not isinstance(loss_decimal_places, int) or isinstance(loss_decimal_places, bool):
        raise TypeError("loss_decimal_places must be an integer")
    if loss_decimal_places < 0:
        raise ValueError("loss_decimal_places must be >= 0")

    if seed is not None:
        random.seed(seed)
        torch.manual_seed(seed)

    if initial_brain is None:
        seed_brain = Ball_Trap_Brain(
            deflectors=deflectors,
            bounce=bounce,
            alphabet=alphabet,
            output=output,
            holes_multiplier=holes_multiplier,
            direction_range=direction_range,
        )
    else:
        seed_brain = initial_brain.clone()

    population = [seed_brain]
    while len(population) < population_size:
        population.append(
            seed_brain.clone().Mutate(
                individual_probability=individual_probability,
                probability=mutation_probability,
            )
        )

    best_brain = None
    best_loss = float("inf")
    history = []

    for generation in range(generations):
        scored = [
            (
                relaxed_loss_fitness(
                    brain,
                    test_cases,
                    temperature=temperature,
                    sharpness=sharpness,
                    logit_strength=logit_strength,
                    repeat_penalty_weight=repeat_penalty_weight,
                    loss_decimal_places=loss_decimal_places,
                ),
                brain,
            )
            for brain in population
        ]

        scored.sort(key=lambda item: item[0])
        generation_loss, generation_best = scored[0]

        if generation_loss < best_loss:
            best_loss = generation_loss
            best_brain = generation_best.clone()

        history.append(best_loss)

        if verbose:
            print(
                f"Generation {generation:4d} | "
                f"generation loss = {generation_loss:.{loss_decimal_places}f} | "
                f"best loss = {best_loss:.{loss_decimal_places}f} | "
                f"temperature = {temperature}"
            )

        if best_loss <= target_loss:
            break

        elites = [brain.clone() for _, brain in scored[:elite_count]]
        new_population = elites[:]

        while len(new_population) < population_size:
            parent = random.choice(elites)
            child = parent.clone().Mutate(
                individual_probability=individual_probability,
                probability=mutation_probability,
            )
            new_population.append(child)

        population = new_population

    return best_brain, history


In [ ]:
def exact_match_prformance (brain, training_data):
  number_of_correct = 0
  for training_case in training_data:
    input_value, target = training_case
    output = ''.join(brain.compute_input(input_value))
    if output == target:
      number_of_correct += 1
  return number_of_correct / len(training_data)

## Example configuration

Replace `training_cases` with the input/target pairs you want the GA to learn.

The important settings are `temperature` and `loss_decimal_places`. `temperature` controls the softness of the relaxed fitness calculation, while `loss_decimal_places` controls how precisely loss is rounded before GA selection.


In [ ]:

training_cases = [(f'1+{i}', f'{1+i}') for i in range(10)]

alphabet = [...]
output = [str(i) for i in range(10)] + list("+-*/=() ")


best_brain, loss_history = genetic_algorithm(
    training_cases,
    alphabet=output,
    output=output,
    deflectors=100,
    bounce=4,
    holes_multiplier=2,
    direction_range=10,
    generations=20,
    population_size=100,
    mutation_probability=0.1,
    temperature=0.0000001,   # <-- adjust this
    loss_decimal_places=100,  # <-- measure/round loss to this many decimals
    sharpness=0.75,
    logit_strength=1.0,
    repeat_penalty_weight=0.3,
    elite_count=1,
    seed=0,
)

print("Final best relaxed loss:", loss_history[-1])
print("Exact match performance:", exact_match_prformance(best_brain, training_cases))


Generation    0 | generation loss = 2.3214189634675843620925661525689065456390380859375000000000000000000000000000000000000000000000000000 | best loss = 2.3214189634675843620925661525689065456390380859375000000000000000000000000000000000000000000000000000 | temperature = 1e-07
Generation    1 | generation loss = 1.8492547149843636677957192659960128366947174072265625000000000000000000000000000000000000000000000000 | best loss = 1.8492547149843636677957192659960128366947174072265625000000000000000000000000000000000000000000000000 | temperature = 1e-07
Generation    2 | generation loss = 1.6923176037273619520817646844079717993736267089843750000000000000000000000000000000000000000000000000 | best loss = 1.6923176037273619520817646844079717993736267089843750000000000000000000000000000000000000000000000000 | temperature = 1e-07
Generation    3 | generation loss = 1.6478570126978084875446484147687442600727081298828125000000000000000000000000000000000000000000000000 | best loss = 1.64785701269